# 1 - Data Cleaning/First Preprocessing 

This project is going to start first getting a better understanding of the data and exploring it.

First making sure there is a common structure, and later visualize, explore and connect the data between them

## 1.1 Imports

In [139]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

## 1.2 Charging The Data and Basic Preprocessing

On this dataset we have more than one .csv, so we are going to load them as we explore them, and one by one decide how to or if joining them into a bigger table

In [140]:
folder = Path().resolve().parent
defecto = folder / "data" / "home-credit-default-risk"

### 1.2.1 application_train.csv

We are going to only use this one, as the test version does not have the target and we cant measure if it performs well on that partition.

Besides given that this is the actual loans we want to predict, only the structure is going to be explored, when we have the final full dataset that is when we are going to do the train/test split and start the proper EDA

In [141]:
application = pd.read_csv(defecto / "application_train.csv")

In [142]:
application.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


Let's see how many observations do we have, which is crucial on the type of split we are going to make

In [143]:
print(f"Applications table has {application.shape[0]} rows and {application.shape[1]} columns")

Applications table has 307511 rows and 122 columns


Okey we have a lot of data actually, so we are going to have a train/val/test approach given the vast amount we have.
Also this implies we can use more variables given the 300k rows and not end overfitting the models so easily.

Given there are 122 columns, we are not going to put a dictionary here, you can find the whole descriptions here :
https://www.kaggle.com/c/home-credit-default-risk/data

On this first dive only basic structure modifications are going to be made, for example one-hot encoding on binary variables, we will leave the rest when we have the full dataset joined

### 1.2.1.1 Checking Nulls

First, let's check for null values

In [144]:
pd.set_option('display.max_rows', None)
resume = application.isnull().sum()
resume[resume > 0].sort_values(ascending=False)

COMMONAREA_MEDI                 214865
COMMONAREA_MODE                 214865
COMMONAREA_AVG                  214865
NONLIVINGAPARTMENTS_MODE        213514
NONLIVINGAPARTMENTS_MEDI        213514
NONLIVINGAPARTMENTS_AVG         213514
FONDKAPREMONT_MODE              210295
LIVINGAPARTMENTS_AVG            210199
LIVINGAPARTMENTS_MEDI           210199
LIVINGAPARTMENTS_MODE           210199
FLOORSMIN_MEDI                  208642
FLOORSMIN_MODE                  208642
FLOORSMIN_AVG                   208642
YEARS_BUILD_MODE                204488
YEARS_BUILD_MEDI                204488
YEARS_BUILD_AVG                 204488
OWN_CAR_AGE                     202929
LANDAREA_AVG                    182590
LANDAREA_MEDI                   182590
LANDAREA_MODE                   182590
BASEMENTAREA_MODE               179943
BASEMENTAREA_MEDI               179943
BASEMENTAREA_AVG                179943
EXT_SOURCE_1                    173378
NONLIVINGAREA_MEDI              169682
NONLIVINGAREA_AVG        

We see actually there is a lot of NA, we are going to drop a lot since the vast majority of them are nulls, but we are going to keep that that the null is another way of telling us another type of information

In [145]:
pd.set_option('display.max_rows', 20)

COMMONAREA and NONLIVINGAPARMENTS are going to be directly dropped, since we would have to know how the data was harvested to see if its that simply was not proportionated by the person solicting the loan, or it means something about the home of that person, since we don't know and its mostly null we are going to directly drop it.

FONDKAPREMONT could betdireclty dropped too because its exclusive to a certain geographical place, but its going to be maintained since that null implies that it does not apply to that person, giving that us more information, we are going to leave it as it is, because of the different values, we cant just conclude they are all equal and can convert into FONDKAREMONT applies or not

In [146]:
to_drop = ["COMMONAREA_MEDI","COMMONAREA_MODE","COMMONAREA_AVG","NONLIVINGAPARTMENTS_MODE","NONLIVINGAPARTMENTS_MEDI","NONLIVINGAPARTMENTS_AVG"]
application.drop(to_drop,axis=1,inplace=True)

LIVINGAPARTMENTS nulls could also imply the person is homeless, or data was not provided, or lives somewhere without with no rooms.
So we are going to nust drop it because it does not gives us certain information

In [147]:
application.drop(["LIVINGAPARTMENTS_AVG","LIVINGAPARTMENTS_MEDI","LIVINGAPARTMENTS_MODE"],axis=1,inplace=True)

FLOORSMIN is going to be dropped too because the same reasoning as LIVINGAPARTMENTS, it does not give us certain info about why is that a null

In [148]:
application.drop(["FLOORSMIN_AVG","FLOORSMIN_MEDI","FLOORSMIN_MODE"],axis=1,inplace=True)

YEARSBUILD too because the same reasoning as before

In [149]:
application.drop(["YEARS_BUILD_AVG","YEARS_BUILD_MEDI","YEARS_BUILD_MODE"],axis=1,inplace=True)

Now with OWN_CAR_AGE, nulls could imply they dont have a car, given the vast majority of nulls, and that that info is already conveyed on the variable FLAG_OWN_CAR, we could just use this second variable, since the first one is unusable due to the quantity of nulls. But first let's check if the hypothesis made its true

In [150]:
application[application["FLAG_OWN_CAR"] == "N"]["OWN_CAR_AGE"].isnull().all()

np.True_

The hypothesis was true, the null means they do not have a car, so we are going to preventively code the NaN as zero

In [151]:
application["OWN_CAR_AGE"] = application["OWN_CAR_AGE"].fillna(value=0)

Now with the LANDAREA variables the same reasoning as the variables we dropped before, we are going to drop them

In [152]:
application.drop(["LANDAREA_AVG","LANDAREA_MEDI","LANDAREA_MODE"],axis=1,inplace=True)

We have remaining

In [153]:
resume = application.isnull().sum()
resume[resume >= application.shape[0]*0.5].sort_values(ascending=False)

FONDKAPREMONT_MODE    210295
BASEMENTAREA_AVG      179943
BASEMENTAREA_MODE     179943
BASEMENTAREA_MEDI     179943
EXT_SOURCE_1          173378
                       ...  
ENTRANCES_MODE        154828
LIVINGAREA_AVG        154350
LIVINGAREA_MODE       154350
LIVINGAREA_MEDI       154350
HOUSETYPE_MODE        154297
Length: 22, dtype: int64

Dropped by the same reasoning as before

In [154]:
application.drop(["BASEMENTAREA_AVG","BASEMENTAREA_MEDI","BASEMENTAREA_MODE","NONLIVINGAREA_AVG","NONLIVINGAREA_MODE","NONLIVINGAREA_MEDI","BASEMENTAREA_AVG","BASEMENTAREA_MODE","BASEMENTAREA_MEDI","ELEVATORS_AVG","ELEVATORS_MODE","ELEVATORS_MEDI","WALLSMATERIAL_MODE","APARTMENTS_MODE","APARTMENTS_AVG","APARTMENTS_MEDI","ENTRANCES_MEDI","ENTRANCES_MODE","ENTRANCES_AVG","LIVINGAREA_MODE","LIVINGAREA_MEDI","LIVINGAREA_AVG","HOUSETYPE_MODE","FLOORSMAX_MODE","FLOORSMAX_MEDI","FLOORSMAX_AVG","YEARS_BEGINEXPLUATATION_MODE","YEARS_BEGINEXPLUATATION_AVG","YEARS_BEGINEXPLUATATION_MEDI","TOTALAREA_MODE","EMERGENCYSTATE_MODE"],axis=1,inplace=True)

The AMT_REQ_CREDIT_BUREAU ones means how many times the historial of the client has been revised, a null could imply it was not checked, that is the client did not solicit a loan on the approximate time span of the variable, so we it suits a 0 in this case to be imputed

In [155]:
bureau_cols = [
    'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY',
    'AMT_REQ_CREDIT_BUREAU_WEEK', 'AMT_REQ_CREDIT_BUREAU_MON',
    'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR'
]
application[bureau_cols] = application[bureau_cols].fillna(value=0)

The EXT_SOURCE ones, since they are scores given by external sources, later we are going to see which one we can impute by median or mean, or which have too many nulls to be used

We no have only remaining

In [156]:
resume = application.isnull().sum()
resume[resume > 0].sort_values(ascending=False)

FONDKAPREMONT_MODE          210295
EXT_SOURCE_1                173378
OCCUPATION_TYPE              96391
EXT_SOURCE_3                 60965
NAME_TYPE_SUITE               1292
OBS_30_CNT_SOCIAL_CIRCLE      1021
DEF_60_CNT_SOCIAL_CIRCLE      1021
OBS_60_CNT_SOCIAL_CIRCLE      1021
DEF_30_CNT_SOCIAL_CIRCLE      1021
EXT_SOURCE_2                   660
AMT_GOODS_PRICE                278
AMT_ANNUITY                     12
CNT_FAM_MEMBERS                  2
DAYS_LAST_PHONE_CHANGE           1
dtype: int64

OCCUPATION_TYPE is going to be filled with Unspecified until the proper EDA can be done, because i suspect the occupation plays a big role in someone paying back or not

In [157]:
application["OCCUPATION_TYPE"] = application["OCCUPATION_TYPE"].fillna(value="Unspecified")

Now with the OBS and DEF one we are going to use the hypothesis that simply there weren't any people observed, or that they did not have contact with anyone that had x time delay in payment, on both ways we can impute with zeros

In [158]:
application["OBS_30_CNT_SOCIAL_CIRCLE"] = application["OBS_30_CNT_SOCIAL_CIRCLE"].fillna(value=0)
application["OBS_60_CNT_SOCIAL_CIRCLE"] = application["OBS_60_CNT_SOCIAL_CIRCLE"].fillna(value=0)
application["DEF_30_CNT_SOCIAL_CIRCLE"] = application["DEF_30_CNT_SOCIAL_CIRCLE"].fillna(value=0)
application["DEF_60_CNT_SOCIAL_CIRCLE"] = application["DEF_60_CNT_SOCIAL_CIRCLE"].fillna(value=0)

For the rest we can just simply use the median or mean when we do the proper eda when we join every table and have the final dataset

Finally on the first table we finally have :

In [159]:
print(application.shape[1],"columns")

76 columns


Let's check if there are any duplicated ids

In [160]:
application["SK_ID_CURR"].nunique() == int(len(application["SK_ID_CURR"]))

True

Since there aren't any duplicated we must format the data on one single id, probably having to make aggregate features

### 1.2.2 installments_payments.csv

Let's start with "installments_payments.csv" on which we can explore the historial of other loans and its respective payment, lets change the type of data so we dont consume all the memory

In [161]:
dtypes = {
    'SK_ID_PREV': 'int32',
    'SK_ID_CURR': 'int32',
    'NUM_INSTALMENT_VERSION': 'uint16',
    'NUM_INSTALMENT_NUMBER': 'uint16',
    'DAYS_INSTALMENT': 'float32',
    'DAYS_ENTRY_PAYMENT': 'float32',
    'AMT_INSTALMENT': 'float32',
    'AMT_PAYMENT': 'float32'
}

payments = pd.read_csv(defecto / "installments_payments.csv",dtype=dtypes)

Before diving into it, let's first explore its columns and what do they mean

In [162]:
print("The columns are :",end=" ")
for column in list(payments.columns):
    print(column,end=", ")

The columns are : SK_ID_PREV, SK_ID_CURR, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER, DAYS_INSTALMENT, DAYS_ENTRY_PAYMENT, AMT_INSTALMENT, AMT_PAYMENT, 

In [163]:
payments.head()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1,6,-1180.0,-1187.0,6948.359863,6948.359863
1,1330831,151639,0,34,-2156.0,-2156.0,1716.525024,1716.525024
2,2085231,193053,2,1,-63.0,-63.0,25425.000000,25425.000000
3,2452527,199697,1,3,-2418.0,-2426.0,24350.130859,24350.130859
4,2714724,167756,1,2,-1383.0,-1366.0,2165.040039,2160.584961


Here a little dictionary to understand the meaning of each variable

| Column | Description |
|--------|-------------|
| `SK_ID_CURR` | ID of the current loan (links to main application table) |
| `SK_ID_PREV` | ID of the previous credit in Home Credit |
| `NUM_INSTALMENT_VERSION` | Version of the installment calendar (0 = credit card) |
| `NUM_INSTALMENT_NUMBER` | Installment number (which payment in the sequence) |
| `DAYS_INSTALMENT` | Day the installment was supposed to be paid (relative to application date) |
| `DAYS_ENTRY_PAYMENT` | Day the installment was actually paid (relative to application date) |
| `AMT_INSTALMENT` | Amount that was supposed to be paid for that installment |
| `AMT_PAYMENT` | Amount actually paid for that installment |

First lets explore if there is going to be any type of temporal leakage.

DAYS_INSTALMENT could be used to determine if there is an active loan ongoing besides the actual requested.
But it could happen that DAYS_INSTALMENT > 0, that is that the date to repay the loan is after the solicitude, but DAYS_ENTRY_PAYMENT < 0, which means it was fully repayed before the loan, so its not active anymore

In [164]:
payments.isnull().sum()

SK_ID_PREV                   0
SK_ID_CURR                   0
NUM_INSTALMENT_VERSION       0
NUM_INSTALMENT_NUMBER        0
DAYS_INSTALMENT              0
DAYS_ENTRY_PAYMENT        2905
AMT_INSTALMENT               0
AMT_PAYMENT               2905
dtype: int64

First lets check if we have to group by ID or there is one row per id

In [165]:
print(f"On Payments table we have {payments.shape[0]} rows and {payments.shape[1]} columns")

On Payments table we have 13605401 rows and 8 columns


Seeing the number of rows it is almost sure there are duplicated ids

In [166]:
payments["SK_ID_CURR"].nunique()

339587

There is duplicated, definitely, so we would have to group by the current loan id, to make sure we can join it with the main table, and get some aggregate metrics, some options are :
- Ratio of times client payed less than it should have, for this it is needed variable that keeps track of difference between the established money and the money payed
- Ratio of payments made late
- Ratio of payments not made, for this we need an impayment variable
- Quantity of loans made by the actual client, for this we only need to count different previous loans id
- Total quantity unpaid

In [167]:
payments["DAYS_PAST_DUE"] = (payments["DAYS_ENTRY_PAYMENT"] - payments["DAYS_INSTALMENT"])
payments["DAYS_PAST_DUE"]

0           -7.0
1            0.0
2            0.0
3           -8.0
4           17.0
            ... 
13605396     NaN
13605397     NaN
13605398     NaN
13605399     NaN
13605400     NaN
Name: DAYS_PAST_DUE, Length: 13605401, dtype: float32

On this new variable, negative values indicate the payment has been made early and positive values otherwise.

Given thtat DAYS_INSTALMENT has no nulls, they come from DAYS_ENTRY_PAYMENT which shows us they have not been payed to this day, so we can make a new variable indicating that the current client has another loan active

Let's consider two distinct cases, one when DAYS_INSTALMENT is < 0, so the time to pay expired and DAYS_ENTRY_PAYMENT is null, it counts as and impayed charge.

On other hand, we have when DAYS_INSTALMENT is > 0, and DAYS_ENTRY_PAYMENT is null, so that indicates he has still time to pay it to this day, this is the client has another active loan

And it could be done too a percent of lately paid charges

In [168]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 13605401 entries, 0 to 13605400
Data columns (total 9 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_PREV              int32  
 1   SK_ID_CURR              int32  
 2   NUM_INSTALMENT_VERSION  uint16 
 3   NUM_INSTALMENT_NUMBER   uint16 
 4   DAYS_INSTALMENT         float32
 5   DAYS_ENTRY_PAYMENT      float32
 6   AMT_INSTALMENT          float32
 7   AMT_PAYMENT             float32
 8   DAYS_PAST_DUE           float32
dtypes: float32(5), int32(2), uint16(2)
memory usage: 415.2 MB


We are going to process it on chunks so the memory does not explode

In [169]:
unpaid_array = np.zeros(len(payments), dtype='uint8')
chunk_size = 2_000_000
for start in range(0, len(payments), chunk_size):
    end = start + chunk_size

    days_inst = payments['DAYS_INSTALMENT'].iloc[start:end].values
    days_entry = payments['DAYS_ENTRY_PAYMENT'].iloc[start:end].values
    
    unpaid_array[start:end] = ((days_inst <= 0) & np.isnan(days_entry)).astype('uint8')
payments['UNPAID_INSTALMENT'] = unpaid_array
payments['UNPAID_INSTALMENT']

0           0
1           0
2           0
3           0
4           0
           ..
13605396    1
13605397    1
13605398    1
13605399    1
13605400    1
Name: UNPAID_INSTALMENT, Length: 13605401, dtype: uint8

In [170]:
amt_payment_clean = payments['AMT_PAYMENT'].fillna(0)
payments["PAYMENT_DIFF"] = (payments["AMT_INSTALMENT"] - amt_payment_clean).astype("Float32")
payments["PAYMENT_RATIO"] = amt_payment_clean / (payments['AMT_INSTALMENT'] +1e-5)

In [171]:
payments.head()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT,DAYS_PAST_DUE,UNPAID_INSTALMENT,PAYMENT_DIFF,PAYMENT_RATIO
0,1054186,161674,1,6,-1180.0,-1187.0,6948.359863,6948.359863,-7.0,0,0.0,1.000000
1,1330831,151639,0,34,-2156.0,-2156.0,1716.525024,1716.525024,0.0,0,0.0,1.000000
2,2085231,193053,2,1,-63.0,-63.0,25425.000000,25425.000000,0.0,0,0.0,1.000000
3,2452527,199697,1,3,-2418.0,-2426.0,24350.130859,24350.130859,-8.0,0,0.0,1.000000
4,2714724,167756,1,2,-1383.0,-1366.0,2165.040039,2160.584961,17.0,0,4.455078,0.997942


In [172]:
loan_level = payments.groupby(['SK_ID_CURR', 'SK_ID_PREV']).agg({
    'NUM_INSTALMENT_NUMBER': 'max',
    'AMT_INSTALMENT': ['mean','sum'],
    'DAYS_PAST_DUE': ['max','min','mean'],
    'PAYMENT_DIFF': ['mean','max','sum'],
    'UNPAID_INSTALMENT': 'sum',
    'PAYMENT_RATIO' : ['mean','min','max']
}).reset_index()

In [173]:
loan_level.columns = ['_'.join(col).strip('_') for col in loan_level.columns.values]

In [174]:
loan_level

,SK_ID_CURR,SK_ID_PREV,NUM_INSTALMENT_NUMBER_max,AMT_INSTALMENT_mean,AMT_INSTALMENT_sum,DAYS_PAST_DUE_max,DAYS_PAST_DUE_min,DAYS_PAST_DUE_mean,PAYMENT_DIFF_mean,PAYMENT_DIFF_max,PAYMENT_DIFF_sum,UNPAID_INSTALMENT_sum,PAYMENT_RATIO_mean,PAYMENT_RATIO_min,PAYMENT_RATIO_max
0,100001,1369693,4,7312.725098,2.925090e+04,-6.0,-36.0,-15.500000,0.0,0.0,0.0,0,1.000000,1.000000,1.000000
1,100001,1851984,4,3981.675049,1.194503e+04,11.0,0.0,3.666667,0.0,0.0,0.0,0,1.000000,1.000000,1.000000
2,100002,1038818,19,11559.247070,2.196257e+05,-12.0,-31.0,-20.421053,0.0,0.0,0.0,0,1.000000,1.000000,1.000000
3,100003,1810518,7,164425.343750,1.150977e+06,-3.0,-8.0,-4.428571,0.0,0.0,0.0,0,1.000000,1.000000,1.000000
4,100003,2396755,12,6731.115234,8.077338e+04,-1.0,-14.0,-6.750000,0.0,0.0,0.0,0,1.000000,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
997747,456255,1359084,8,15610.859375,1.404977e+05,7.0,-22.0,-4.222222,1257.130005,9248.219727,11314.169922,0,0.888889,0.182598,1.000000
997748,456255,1743609,10,13218.228516,1.321823e+05,5.0,-14.0,-6.400000,0.0,0.0,0.0,0,1.000000,1.000000,1.000000
997749,456255,2073384,3,63086.625000,3.154331e+05,5.0,-30.0,-12.600000,6560.243652,16365.644531,32801.21875,0,0.600000,0.002132,1.000000
997750,456255,2631384,24,58353.824219,1.692261e+06,4.0,-35.0,-9.827586,-19285.960938,27378.808594,-559292.875,0,1.257803,0.004034,12.388469


In [175]:
client_installments_agg = loan_level.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',  # How many loans has the client made
    'AMT_INSTALMENT_sum': ['mean'], # Mean of amounts of the loans the client made
    'NUM_INSTALMENT_NUMBER_max': 'max', # How much cuotas
    'AMT_INSTALMENT_mean': ['mean', 'max'], # How much does the client pay on mean and max
    'DAYS_PAST_DUE_max': 'max',
    'DAYS_PAST_DUE_mean': 'mean', 
    'PAYMENT_RATIO_mean':'mean',
    'PAYMENT_RATIO_min': 'min',
    'PAYMENT_RATIO_max': 'max',
    'PAYMENT_DIFF_sum': ['mean','sum'],
    'PAYMENT_DIFF_max':'max',
    'UNPAID_INSTALMENT_sum':'sum'

})

client_installments_agg.columns = ['INS_' + '_'.join(col).upper() for col in client_installments_agg.columns]
client_installments_agg = client_installments_agg.reset_index()

In [176]:
client_installments_agg.head()

,SK_ID_CURR,INS_SK_ID_PREV_COUNT,INS_AMT_INSTALMENT_SUM_MEAN,INS_NUM_INSTALMENT_NUMBER_MAX_MAX,INS_AMT_INSTALMENT_MEAN_MEAN,INS_AMT_INSTALMENT_MEAN_MAX,INS_DAYS_PAST_DUE_MAX_MAX,INS_DAYS_PAST_DUE_MEAN_MEAN,INS_PAYMENT_RATIO_MEAN_MEAN,INS_PAYMENT_RATIO_MIN_MIN,INS_PAYMENT_RATIO_MAX_MAX,INS_PAYMENT_DIFF_SUM_MEAN,INS_PAYMENT_DIFF_SUM_SUM,INS_PAYMENT_DIFF_MAX_MAX,INS_UNPAID_INSTALMENT_SUM_SUM
0,100001,2,20597.962891,4,5647.200195,7312.725098,11.0,-5.916667,1.0,1.0,1.0,0.0,0.0,0.0,0
1,100002,1,219625.703125,19,11559.247070,11559.247070,-12.0,-20.421053,1.0,1.0,1.0,0.0,0.0,0.0,0
2,100003,3,539621.562500,12,78558.476562,164425.343750,-1.0,-7.448412,1.0,1.0,1.0,0.0,0.0,0.0,0
3,100004,1,21288.464844,3,7096.154785,7096.154785,-3.0,-7.666667,1.0,1.0,1.0,0.0,0.0,0.0,0
4,100005,1,56161.843750,9,6240.205078,6240.205078,1.0,-23.555555,1.0,1.0,1.0,0.0,0.0,0.0,0


With this we have all the metrics summarized and organized by the id of the current loan, so we will conclude on this table and keep formatting the rest of them

## 1.2.3 previous_application.csv

On this table we have data about all the request made, accepted or not by the client oh Home Credit

In [177]:
previous = pd.read_csv(defecto / "previous_application.csv")

In [178]:
previous.head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [179]:
print(f"On the previous_application table we have {previous.shape[0]} rows and {previous.shape[1]} columns")

On the previous_application table we have 1670214 rows and 37 columns


In [180]:
previous['CODE_REJECT_REASON'].value_counts()

CODE_REJECT_REASON
XAP       1353093
HC         175231
LIMIT       55680
SCO         37467
CLIENT      26436
SCOFR       12811
XNA          5244
VERIF        3535
SYSTEM        717
Name: count, dtype: int64

Here is a dictionary explaining briefly the meaning of every column

| Column Name | Category | Description |
| :--- | :--- | :--- |
| `SK_ID_PREV` | Identifier | Unique ID of previous application / loan at Home Credit. |
| `SK_ID_CURR` | Identifier | Unique ID of current loan applicant. |
| `NAME_CONTRACT_TYPE` | Contract / Status | Contract product type (Consumer loans, Cash loans, Revolving loans). |
| `NAME_CONTRACT_STATUS` | Contract / Status | Final status of previous application (Approved, Refused, Canceled, Unused offer). |
| `CODE_REJECT_REASON` | Contract / Status | Reason why the application was refused (SCO, LIMIT, HC, CLIENT, SYSTEM, etc.). |
| `AMT_APPLICATION` | Financial Amount | Credit amount requested by the client in the previous application. |
| `AMT_CREDIT` | Financial Amount | Final credit amount approved and granted by the bank. |
| `AMT_ANNUITY` | Financial Amount | Monthly installment amount of the previous application. |
| `AMT_DOWN_PAYMENT` | Financial Amount | Down payment amount paid by the client on the previous application. |
| `AMT_GOODS_PRICE` | Financial Amount | Price of the goods that the client asked to finance. |
| `RATE_DOWN_PAYMENT` | Financial Ratio | Down payment rate normalized relative to the price of the goods. |
| `RATE_INTEREST_PRIMARY` | Financial Ratio | Primary interest rate (~100% missing values). |
| `RATE_INTEREST_PRIVILEGED` | Financial Ratio | Privileged interest rate (~100% missing values). |
| `CNT_PAYMENT` | Financial Terms | Term of previous credit (number of monthly installments). |
| `NAME_CASH_LOAN_PURPOSE` | Business / Product | Purpose of the cash loan (Repairs, Education, Furniture, Urgent, etc.). |
| `NAME_CLIENT_TYPE` | Business / Product | Client status when applying (New, Repeater, Refreshed). |
| `NAME_PAYMENT_TYPE` | Business / Product | Agreed payment method (Cash through bank, Cashless, etc.). |
| `NAME_TYPE_SUITE` | Business / Product | Who accompanied the client when applying (Unaccompanied, Family, etc.). |
| `NAME_GOODS_CATEGORY` | Business / Product | Category of the goods financed (Mobile, Furniture, Audio/Video, Computers, etc.). |
| `NAME_PORTFOLIO` | Business / Product | Portfolio category of the previous application (Cards, POS, Cash, Cars). |
| `NAME_PRODUCT_TYPE` | Business / Product | Product sales type (x-sell = cross-sell, walk-in = new client). |
| `CHANNEL_TYPE` | Business / Product | Acquisition channel used for application (Credit Express, Country-wide, Regional, etc.). |
| `SELLERPLACE_AREA` | Business / Product | Commercial surface area (m²) of the seller place. |
| `NAME_SELLERPLACE_INDUSTRY` | Business / Product | Seller industry sector (Connectivity, Furniture, Consumer Electronics, etc.). |
| `NAME_YIELD_GROUP` | Business / Product | Interest yield group of the loan (high, middle, low_normal, low_action). |
| `PRODUCT_COMBINATION` | Business / Product | Commercial product package combination name. |
| `DAYS_DECISION` | Date / Time | Days since decision was made relative to current application date (negative days). |
| `DAYS_FIRST_DRAWING` | Date / Time | Relative day when the first disbursement was drawn. |
| `DAYS_FIRST_DUE` | Date / Time | Relative day when the first installment was scheduled to be due. |
| `DAYS_LAST_DUE_1ST_VERSION` | Date / Time | Relative day when the first version of the last installment was due. |
| `DAYS_LAST_DUE` | Date / Time | Relative day when the last installment was actually paid. |
| `DAYS_TERMINATION` | Date / Time | Relative day when the previous contract officially ended/liquidated. |
| `WEEKDAY_APPR_PROCESS_START` | Flags / Misc | Day of the week when the previous application started. |
| `HOUR_APPR_PROCESS_START` | Flags / Misc | Hour of the day when the previous application started. |
| `FLAG_LAST_APPL_PER_CONTRACT` | Flags / Misc | Flag if this was the last application for the contract (Y/N). |
| `NFLAG_LAST_APPL_IN_DAY` | Flags / Misc | Flag if this was the last application of the client on that day (1/0). |
| `NFLAG_INSURED_ON_APPROVAL` | Flags / Misc | Flag if the client requested insurance during application approval (1/0). |

First, let's treat the null values, and see if they are going to be dropped or codified

In [181]:
nulls = previous.isnull().sum().sort_values(ascending=False)
nulls[nulls > 0]

RATE_INTEREST_PRIVILEGED     1664263
RATE_INTEREST_PRIMARY        1664263
AMT_DOWN_PAYMENT              895844
RATE_DOWN_PAYMENT             895844
NAME_TYPE_SUITE               820405
DAYS_TERMINATION              673065
DAYS_FIRST_DRAWING            673065
DAYS_FIRST_DUE                673065
DAYS_LAST_DUE_1ST_VERSION     673065
DAYS_LAST_DUE                 673065
NFLAG_INSURED_ON_APPROVAL     673065
AMT_GOODS_PRICE               385515
AMT_ANNUITY                   372235
CNT_PAYMENT                   372230
PRODUCT_COMBINATION              346
AMT_CREDIT                         1
dtype: int64

The RATE_INTEREST_PRIVILEGED and RATE_INTEREST_PRIMARY are the most critical here

In [182]:
print(f"Both columns have respectively {nulls["RATE_INTEREST_PRIVILEGED"]/previous.shape[0]*100:.4f}% nulls and {nulls["RATE_INTEREST_PRIMARY"]/previous.shape[0]*100:.4f}% nulls")

Both columns have respectively 99.6437% nulls and 99.6437% nulls


The only choice is to see if nulls convey some type of information, because with 99% we can't impute any value, it would add a lot of bias to our data.

Looking for documentation, the reason found was that they were special type of interests that were not usually registered, so we will just drop them

In [183]:
previous.drop(["RATE_INTEREST_PRIVILEGED","RATE_INTEREST_PRIMARY"],axis=1,inplace=True)

Now treating AMT_DOWN_PAYMENT and RATE_DOWN_PAYMENT, they both have the same type of nulls, are they the same ones?

In [184]:
print(previous[previous["AMT_DOWN_PAYMENT"].isnull()]["RATE_DOWN_PAYMENT"].isnull().all())
print(previous[previous["RATE_DOWN_PAYMENT"].isnull()]["AMT_DOWN_PAYMENT"].isnull().all())

True
True


Ok, so a pattern can be seen that relates these two types of nulls, searching about these variables results that on cash loans or without a fee, dont have an initial pay, that means they payed 0 monetary units, so we will just impute with 0

In [185]:
previous["PAYMENT_MISSING"] = previous["AMT_DOWN_PAYMENT"].isnull().astype(int)
previous["AMT_DOWN_PAYMENT"] = previous["AMT_DOWN_PAYMENT"].fillna(value=0)
previous["RATE_DOWN_PAYMENT"] = previous["RATE_DOWN_PAYMENT"].fillna(value=0)

Interestingly it is shown that all the DAYS variables have the same number of nulls, let's check if they are all the same rows

In [186]:
previous[previous["DAYS_TERMINATION"].isnull()][["DAYS_FIRST_DRAWING","DAYS_FIRST_DUE","DAYS_LAST_DUE_1ST_VERSION","DAYS_LAST_DUE","NFLAG_INSURED_ON_APPROVAL"]].isnull().all()

DAYS_FIRST_DRAWING           True
DAYS_FIRST_DUE               True
DAYS_LAST_DUE_1ST_VERSION    True
DAYS_LAST_DUE                True
NFLAG_INSURED_ON_APPROVAL    True
dtype: bool

Yep, they are all the same rows, so given the pattern and the nature of the variables, it must be that the loans were not accepted, we will not keep them because half of the meaning of the info is that the loan was not accepted, which we already have on other variables

In [187]:
previous.drop(["DAYS_FIRST_DRAWING","DAYS_FIRST_DUE","DAYS_LAST_DUE_1ST_VERSION","DAYS_LAST_DUE","DAYS_TERMINATION","NFLAG_INSURED_ON_APPROVAL"],axis=1,inplace=True)

The AMT_ANNUITY nulls, given that it is the monthly installment of the previous application could mean, that the previous loan was not accepted or it didn't had a fixed annuity

In [188]:
previous[previous["AMT_ANNUITY"].isnull()]["NAME_CONTRACT_STATUS"].value_counts()

NAME_CONTRACT_STATUS
Canceled        305805
Refused          40898
Unused offer     25524
Approved             8
Name: count, dtype: int64

It is going to be filled with zeros, as those loans never got active

In [189]:
previous["AMT_ANNUITY"] = previous["AMT_ANNUITY"].fillna(value=0)

In [190]:
nulls = previous.isnull().sum().sort_values(ascending=False)
nulls[nulls > 0]

NAME_TYPE_SUITE        820405
AMT_GOODS_PRICE        385515
CNT_PAYMENT            372230
PRODUCT_COMBINATION       346
AMT_CREDIT                  1
dtype: int64

NAME_TYPE_SUITE is mostly nulls so that info was not took, and besides who accompanies the client does not have predictive potential, given all the other variables we have, so its just mostly noise

In [191]:
previous.drop("NAME_TYPE_SUITE",axis=1,inplace=True)

AMT_GOODS_PRICE is going to be dropped. As we can use AMT_APPLICATION and AMT_CREDIT to make a measure of trust, calculating the ratio between how much money was asked and how much the client received, if the client received less than asked it is an indicator of distrust, but if he received more than he asked it is measure that they trust the client can pay even more than it asked to. And drop the variable because of the large number of nulls

In [192]:
previous["ASK_GIVEN_RATIO"] = previous["AMT_CREDIT"] / previous["AMT_APPLICATION"]
previous.drop("AMT_GOODS_PRICE",axis=1,inplace=True)

In [193]:
previous.head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,FLAG_LAST_APPL_PER_CONTRACT,...,NAME_PORTFOLIO,NAME_PRODUCT_TYPE,CHANNEL_TYPE,SELLERPLACE_AREA,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,PAYMENT_MISSING,ASK_GIVEN_RATIO
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,SATURDAY,15,Y,...,POS,XNA,Country-wide,35,Connectivity,12.0,middle,POS mobile with interest,0,1.00000
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,0.0,THURSDAY,11,Y,...,Cash,x-sell,Contact center,-1,XNA,36.0,low_action,Cash X-Sell: low,1,1.11880
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,0.0,TUESDAY,11,Y,...,Cash,x-sell,Credit and cash offices,-1,XNA,12.0,high,Cash X-Sell: high,1,1.21284
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,0.0,MONDAY,7,Y,...,Cash,x-sell,Credit and cash offices,-1,XNA,12.0,middle,Cash X-Sell: middle,1,1.04620
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,0.0,THURSDAY,9,Y,...,Cash,walk-in,Credit and cash offices,-1,XNA,24.0,high,Cash Street: high,1,1.19720


In [194]:
# dropped because its redundant
previous.drop("PRODUCT_COMBINATION",axis=1,inplace=True)

Let's prepare some final variables, to make the final aggregate variables table

In [195]:
previous['IS_APPROVED'] = (previous['NAME_CONTRACT_STATUS'] == 'Approved').astype('uint8')
previous['IS_REFUSED'] = (previous['NAME_CONTRACT_STATUS'] == 'Refused').astype('uint8')
previous['IS_CANCELED'] = (previous['NAME_CONTRACT_STATUS'] == 'Canceled').astype('uint8')

In [196]:
prev_agg = previous.groupby('SK_ID_CURR').agg({

    'SK_ID_PREV': 'count',

    'IS_APPROVED': 'sum',
    'IS_REFUSED': 'sum',
    'IS_CANCELED': 'sum',

    'AMT_CREDIT': ['mean', 'max'],
    'AMT_ANNUITY': ['mean', 'max'],
    'CNT_PAYMENT': ['mean', 'max'],
    'ASK_GIVEN_RATIO': ['mean', 'min', 'max']
})
# Aplanar nombres de columnas
prev_agg.columns = ['PREV_' + '_'.join(col).upper() for col in prev_agg.columns]
prev_agg = prev_agg.reset_index()

In [197]:
prev_sorted = previous.sort_values(['SK_ID_CURR', 'DAYS_DECISION'], ascending=[True, False])

last_3 = prev_sorted.groupby('SK_ID_CURR').head(3)
approval_last_3 = last_3.groupby('SK_ID_CURR')['IS_APPROVED'].mean().rename('PREV_APPROVAL_RATE_LAST_3')

last_5 = prev_sorted.groupby('SK_ID_CURR').head(5)
approval_last_5 = last_5.groupby('SK_ID_CURR')['IS_APPROVED'].mean().rename('PREV_APPROVAL_RATE_LAST_5')

In [198]:
prev_agg = prev_agg.merge(approval_last_3, on='SK_ID_CURR', how='left')
prev_agg = prev_agg.merge(approval_last_5, on='SK_ID_CURR', how='left')

In [199]:
prev_agg.isnull().sum()

SK_ID_CURR                     0
PREV_SK_ID_PREV_COUNT          0
PREV_IS_APPROVED_SUM           0
PREV_IS_REFUSED_SUM            0
PREV_IS_CANCELED_SUM           0
PREV_AMT_CREDIT_MEAN           0
PREV_AMT_CREDIT_MAX            0
PREV_AMT_ANNUITY_MEAN          0
PREV_AMT_ANNUITY_MAX           0
PREV_CNT_PAYMENT_MEAN        478
PREV_CNT_PAYMENT_MAX         478
PREV_ASK_GIVEN_RATIO_MEAN    252
PREV_ASK_GIVEN_RATIO_MIN     252
PREV_ASK_GIVEN_RATIO_MAX     252
PREV_APPROVAL_RATE_LAST_3      0
PREV_APPROVAL_RATE_LAST_5      0
dtype: int64

In [201]:
prev_agg['PREV_REFUSED_RATIO'] = prev_agg['PREV_IS_REFUSED_SUM'] / prev_agg['PREV_SK_ID_PREV_COUNT']
prev_agg['PREV_APPROVED_RATIO'] = prev_agg['PREV_IS_APPROVED_SUM'] / prev_agg['PREV_SK_ID_PREV_COUNT']

In [202]:
prev_agg.head()

,SK_ID_CURR,PREV_SK_ID_PREV_COUNT,PREV_IS_APPROVED_SUM,PREV_IS_REFUSED_SUM,PREV_IS_CANCELED_SUM,PREV_AMT_CREDIT_MEAN,PREV_AMT_CREDIT_MAX,PREV_AMT_ANNUITY_MEAN,PREV_AMT_ANNUITY_MAX,PREV_CNT_PAYMENT_MEAN,PREV_CNT_PAYMENT_MAX,PREV_ASK_GIVEN_RATIO_MEAN,PREV_ASK_GIVEN_RATIO_MIN,PREV_ASK_GIVEN_RATIO_MAX,PREV_APPROVAL_RATE_LAST_3,PREV_APPROVAL_RATE_LAST_5,PREV_REFUSED_RATIO,PREV_APPROVED_RATIO
0,100001,1,1,0,0,23787.00,23787.0,3951.000,3951.000,8.0,8.0,0.957782,0.957782,0.957782,1.0,1.0,0.0,1.0
1,100002,1,1,0,0,179055.00,179055.0,9251.775,9251.775,24.0,24.0,1.000000,1.000000,1.000000,1.0,1.0,0.0,1.0
2,100003,3,3,0,0,484191.00,1035882.0,56553.990,98356.995,10.0,12.0,1.057664,0.989013,1.150980,1.0,1.0,0.0,1.0
3,100004,1,1,0,0,20106.00,20106.0,5357.250,5357.250,4.0,4.0,0.828021,0.828021,0.828021,1.0,1.0,0.0,1.0
4,100005,2,1,0,1,20076.75,40153.5,2406.600,4813.200,12.0,12.0,0.899950,0.899950,0.899950,0.5,0.5,0.0,0.5


## 1.2.4 bureau.csv

On this new table we have the historial of the clients on another entities, so at first glance its a very important one

Let's do some basic exploring first

In [204]:
bureau = pd.read_csv(defecto / "bureau.csv")

In [205]:
bureau.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [206]:
print(f"This table has {bureau.shape[0]} rows and {bureau.shape[1]} columns")

This table has 1716428 rows and 17 columns


No repeated bureau ids here

In [207]:
bureau["SK_ID_BUREAU"].nunique()

1716428

Let's look for potential leakage, loans made after the request we received

In [209]:
bureau[bureau["DAYS_CREDIT"] > 0].count()

SK_ID_CURR                0
SK_ID_BUREAU              0
CREDIT_ACTIVE             0
CREDIT_CURRENCY           0
DAYS_CREDIT               0
CREDIT_DAY_OVERDUE        0
DAYS_CREDIT_ENDDATE       0
DAYS_ENDDATE_FACT         0
AMT_CREDIT_MAX_OVERDUE    0
CNT_CREDIT_PROLONG        0
AMT_CREDIT_SUM            0
AMT_CREDIT_SUM_DEBT       0
AMT_CREDIT_SUM_LIMIT      0
AMT_CREDIT_SUM_OVERDUE    0
CREDIT_TYPE               0
DAYS_CREDIT_UPDATE        0
AMT_ANNUITY               0
dtype: int64

So this means not a single loan here was made after the current loan, so by this part we wont have any leakage

Here is a dictionary to get a better understanding of what every column means

| Column | Category | Description |
| :--- | :--- | :--- |
| `SK_ID_CURR` | Identifier | Unique ID of the current loan applicant. |
| `SK_ID_BUREAU` | Identifier | Unique ID of the credit record in the bureau. |
| `CREDIT_ACTIVE` | Status | Status of the external credit (Active, Closed, Sold, Bad debt). |
| `CREDIT_CURRENCY` | Status | Currency of the external credit. |
| `DAYS_CREDIT` | Date / Time | Days since the bureau credit was opened (negative, relative to application date). |
| `CREDIT_DAY_OVERDUE` | Risk | Number of days the client is currently overdue on the bureau credit. |
| `DAYS_CREDIT_ENDDATE` | Date / Time | Remaining duration of the bureau credit (in days). Positive = still active, negative = already ended. |
| `DAYS_ENDDATE_FACT` | Date / Time | Actual end date of the credit (only for closed credits). |
| `AMT_CREDIT_MAX_OVERDUE` | Risk | Maximum amount the client was ever overdue on this credit. |
| `CNT_CREDIT_PROLONG` | Risk | Number of times the credit was prolonged/restructured. |
| `AMT_CREDIT_SUM` | Financial | Total credit amount of the bureau credit. |
| `AMT_CREDIT_SUM_DEBT` | Financial | Current outstanding debt on the bureau credit. |
| `AMT_CREDIT_SUM_LIMIT` | Financial | Credit limit on the bureau credit (for credit cards/revolving). |
| `AMT_CREDIT_SUM_OVERDUE` | Financial | Current overdue amount on the bureau credit. |
| `CREDIT_TYPE` | Type | Type of bureau credit (Consumer credit, Credit card, Mortgage, Car loan, etc.). |
| `DAYS_CREDIT_UPDATE` | Date / Time | Days since the last update of the bureau credit information. |
| `AMT_ANNUITY` | Financial | Monthly annuity/installment of the bureau credit. |

Creating some variables about the status, momentarily considering the NA as positive values as they mean the loan does not have a fixed date of completion

In [210]:
bureau['IS_ACTIVE_EXTERNAL'] = (bureau['CREDIT_ACTIVE'] == 'Active').astype('uint8')
bureau["IS_OVERDUE"] = ((bureau['CREDIT_ACTIVE'] == 'Active') & (bureau["DAYS_CREDIT_ENDDATE"].fillna(365) < 0)).astype('uint8')
bureau['HAS_BAD_DEBT'] = (bureau['CREDIT_ACTIVE'] == 'Bad debt').astype('uint8')
bureau['HAS_SOLD_DEBT'] = (bureau['CREDIT_ACTIVE'] == 'Sold').astype('uint8')
bureau['CREDIT_UTILIZATION'] = bureau['AMT_CREDIT_SUM_DEBT'] / (bureau['AMT_CREDIT_SUM_LIMIT'].replace(0, np.nan))

In [211]:
bureau.isnull().sum().sort_values(ascending=False)

CREDIT_UTILIZATION        1642022
AMT_ANNUITY               1226791
AMT_CREDIT_MAX_OVERDUE    1124488
DAYS_ENDDATE_FACT          633653
AMT_CREDIT_SUM_LIMIT       591780
                           ...   
CREDIT_TYPE                     0
IS_ACTIVE_EXTERNAL              0
IS_OVERDUE                      0
HAS_BAD_DEBT                    0
HAS_SOLD_DEBT                   0
Length: 22, dtype: int64

AMT_ANNUITY has the most nulls, but similarly to last variables, it does not have a defined cuota per time, so the null is giving us info but the rate of nulls is too high and we can estimate it with other variables that dont have that much nulls

In [212]:
bureau.drop("AMT_ANNUITY",axis=1,inplace=True)

As well as AMT_CREDIT_MAX_OVERDUE due to the high rate of nulls, and part of the info is already on another variables like AMT_SUM_CREDIT_OVERDUE and CREDIT_DAY_OVERDUE, DAYS_ENDDATE_FACT as the high rate of nulls and its null info is already on the new variables created

In [213]:
bureau.drop(["AMT_CREDIT_MAX_OVERDUE","DAYS_ENDDATE_FACT"],axis=1,inplace=True)

Now with AMT_CREDIT_SUM_DEBT a null implies the loan was closed without any debt, so we will fill it with zeros

In [ ]:
bureau["AMT_CREDIT_SUM_DEBT"] = bureau["AMT_CREDIT_SUM_DEBT"].fillna(0)

The sum limit we will leave it as it is, because it means there is no limit, so we dont insert any value that morphs the metrics as groupby already ignores the NA

And drop the enddate, because its info is on the new variables we created before

In [215]:
bureau.drop("DAYS_CREDIT_ENDDATE",axis=1,inplace=True)

In [216]:
bureau.isnull().sum().sort_values(ascending=False)

CREDIT_UTILIZATION        1642022
AMT_CREDIT_SUM_LIMIT       591780
AMT_CREDIT_SUM                 13
SK_ID_CURR                      0
SK_ID_BUREAU                    0
CREDIT_ACTIVE                   0
CREDIT_DAY_OVERDUE              0
DAYS_CREDIT                     0
CNT_CREDIT_PROLONG              0
CREDIT_CURRENCY                 0
AMT_CREDIT_SUM_DEBT             0
AMT_CREDIT_SUM_OVERDUE          0
DAYS_CREDIT_UPDATE              0
CREDIT_TYPE                     0
IS_ACTIVE_EXTERNAL              0
IS_OVERDUE                      0
HAS_BAD_DEBT                    0
HAS_SOLD_DEBT                   0
dtype: int64

Now that is has been cleaned, tha aggregate variables have to be made

In [221]:
bureau_agg = bureau.groupby('SK_ID_CURR').agg({
    'SK_ID_BUREAU': 'count',
    
    'IS_ACTIVE_EXTERNAL': 'sum',
    'IS_OVERDUE': 'sum',
    'HAS_BAD_DEBT': 'sum',
    'HAS_SOLD_DEBT': 'sum',
    'DAYS_CREDIT': 'max',
    
    'AMT_CREDIT_SUM': ['mean', 'max'],
    'AMT_CREDIT_SUM_DEBT': ['sum', 'mean', 'max'],
    
    'CREDIT_DAY_OVERDUE': ['mean', 'max'],
    'AMT_CREDIT_SUM_OVERDUE': ['sum', 'mean'],
    'CNT_CREDIT_PROLONG': ['mean', 'max'],
    
    'CREDIT_UTILIZATION': ['mean', 'max']
}).reset_index()

In [222]:
bureau_agg.columns = ['BUREAU_' + '_'.join(col).upper().strip('_') 
                      if col[0] != 'SK_ID_CURR' else 'SK_ID_CURR' 
                      for col in bureau_agg.columns]

In [223]:
bureau_agg

,SK_ID_CURR,BUREAU_SK_ID_BUREAU_COUNT,BUREAU_IS_ACTIVE_EXTERNAL_SUM,BUREAU_IS_OVERDUE_SUM,BUREAU_HAS_BAD_DEBT_SUM,BUREAU_HAS_SOLD_DEBT_SUM,BUREAU_DAYS_CREDIT_MAX,BUREAU_AMT_CREDIT_SUM_MEAN,BUREAU_AMT_CREDIT_SUM_MAX,BUREAU_AMT_CREDIT_SUM_DEBT_SUM,BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,BUREAU_AMT_CREDIT_SUM_DEBT_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM,BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN,BUREAU_CNT_CREDIT_PROLONG_MEAN,BUREAU_CNT_CREDIT_PROLONG_MAX,BUREAU_CREDIT_UTILIZATION_MEAN,BUREAU_CREDIT_UTILIZATION_MAX
0,100001,7,3,0,0,0,-49,2.076236e+05,378000.00,596686.500,85240.928571,373239.00,0.0,0,0.0,0.0,0.000000,0,NaN,NaN
1,100002,8,2,0,0,0,-103,1.081319e+05,450000.00,245781.000,30722.625000,245781.00,0.0,0,0.0,0.0,0.000000,0,0.000000,0.000000
2,100003,4,1,0,0,0,-606,2.543501e+05,810000.00,0.000,0.000000,0.00,0.0,0,0.0,0.0,0.000000,0,0.000000,0.000000
3,100004,2,0,0,0,0,-408,9.451890e+04,94537.80,0.000,0.000000,0.00,0.0,0,0.0,0.0,0.000000,0,NaN,NaN
4,100005,3,2,0,0,0,-62,2.190420e+05,568800.00,568408.500,189469.500000,543087.00,0.0,0,0.0,0.0,0.000000,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305806,456249,13,2,1,0,0,-483,2.841430e+05,765000.00,163071.000,12543.923077,163071.00,0.0,0,0.0,0.0,0.000000,0,NaN,NaN
305807,456250,3,2,0,0,0,-760,1.028820e+06,2153110.05,2232040.095,744013.365000,1840308.48,0.0,0,0.0,0.0,0.000000,0,6.722884,6.722884
305808,456253,4,2,0,0,0,-713,9.900000e+05,2250000.00,1795833.000,448958.250000,1624797.00,0.0,0,0.0,0.0,0.000000,0,NaN,NaN
305809,456254,1,0,0,0,0,-1104,4.500000e+04,45000.00,0.000,0.000000,0.00,0.0,0,0.0,0.0,0.000000,0,NaN,NaN


## 1.2.5 credit_card_balance.csv

The observations consists of monthly snapshots of previous credit cards issued by Home Credit

In [224]:
card_balance = pd.read_csv(defecto / "credit_card_balance.csv")

In [225]:
card_balance.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,...,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,...,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0


In [226]:
print(f"This table has {card_balance.shape[0]} observations and {card_balance.shape[1]} columns")

This table has 3840312 observations and 23 columns


| Column Name | Description |
| :--- | :--- |
| **`SK_ID_PREV`** | ID of previous credit card contract in Home Credit. |
| **`SK_ID_CURR`** | ID of current application loan in sample. |
| **`MONTHS_BALANCE`** | Month of balance relative to application date (-1 means latest month). |
| **`AMT_BALANCE`** | Total outstanding balance on the credit card during the month. |
| **`AMT_CREDIT_LIMIT_ACTUAL`** | Credit card limit during the month. |
| **`AMT_DRAWINGS_ATM_CURRENT`** | Amount drawn at ATM using the credit card during the month. |
| **`AMT_DRAWINGS_CURRENT`** | Total amount drawn (spent/withdrawn) during the month. |
| **`AMT_DRAWINGS_OTHER_CURRENT`** | Amount drawn in other transactions during the month. |
| **`AMT_DRAWINGS_POS_CURRENT`** | Amount drawn for POS purchases during the month. |
| **`AMT_INST_MIN_REGULARITY`** | Minimum required installment payment for the month. |
| **`AMT_PAYMENT_CURRENT`** | Amount paid by the client during the month. |
| **`AMT_PAYMENT_TOTAL_CURRENT`** | Total amount paid by the client on the credit card during the month. |
| **`AMT_RECEIVABLE_PRINCIPAL`** | Principal receivable amount on the credit card. |
| **`AMT_RECIVABLE`** | Total receivable amount on the credit card. |
| **`AMT_TOTAL_RECEIVABLE`** | Total amount receivable on the credit card. |
| **`CNT_DRAWINGS_ATM_CURRENT`** | Number of ATM withdrawals made during the month. |
| **`CNT_DRAWINGS_CURRENT`** | Total number of drawing transactions made during the month. |
| **`CNT_DRAWINGS_OTHER_CURRENT`** | Number of other drawing transactions made during the month. |
| **`CNT_DRAWINGS_POS_CURRENT`** | Number of POS purchases made during the month. |
| **`CNT_INSTALMENT_MATURE_CUM`** | Cumulative number of completed/matured installment payments. |
| **`NAME_CONTRACT_STATUS`** | Contract status during the month (e.g., Active, Completed, Signed). |
| **`SK_DPD`** | Days Past Due (DPD) during the month. |
| **`SK_DPD_DEF`** | Days Past Due during the month ignoring minimal tolerance. |